# SV with triple exogenous regressors (volume/money/avg)

本 notebook 展示：针对 11 个合约，同时在状态方程中加入经过对数+标准化后的 volume、money、avg 三个协变量，运行带外生变量的 AR(1) 随机波动模型。

In [ ]:
# 导入依赖，注释使用中文
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sv_toolkit.data import list_csv_files, load_single_file, get_contract_symbol_from_path
from sv_toolkit.batch import make_timestamped_root, save_param_summary
from sv_toolkit.mcmc import run_mcmc_sv
from sv_toolkit.exog_multi import prepare_multi_exog
from sv_toolkit.plotting import (
    plot_volatility,
    plot_param_posterior,
    plot_standardized_residuals,
    plot_mixture_usage,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

data_dir = Path('../2005年__20250905')
output_root = make_timestamped_root(Path('outputs'))
experiment_dir = output_root / 'exog_triplet'
experiment_dir.mkdir(parents=True, exist_ok=True)

print(f"Environment ready. Data dir: {data_dir}. Output root: {output_root}")

## 1. 选择需要处理的 11 个合约 CSV

In [ ]:
target_files = [
    "AG_主力合约_1m数据.csv",
    "AL_主力合约_1m数据.csv",
    "AP_主力合约_1m数据.csv",
    "AU_主力合约_1m数据.csv",
    "BB_主力合约_1m数据.csv",
    "BU_主力合约_1m数据.csv",
    "CF_主力合约_1m数据.csv",
    "CJ_主力合约_1m数据.csv",
    "CS_主力合约_1m数据.csv",
    "CU_主力合约_1m数据.csv",
    "CY_主力合约_1m数据.csv",
]

all_csv = list_csv_files(data_dir, max_files=300)
name_to_path = {p.name: p for p in all_csv}

selected_paths = []
for name in target_files:
    if name not in name_to_path:
        print(f"Warning: {name} not found under {data_dir}")
    else:
        selected_paths.append(name_to_path[name])

if not selected_paths:
    raise RuntimeError("None of the target csv files were found in the data directory.")

print("Selected paths:")
for p in selected_paths:
    print(" -", p)

## 2. 循环处理每个合约：加载、提取三维外生变量、运行 MCMC、保存输出

In [ ]:
datasets = {}

for file_path in selected_paths:
    symbol = get_contract_symbol_from_path(file_path)
    contract_out_dir = experiment_dir / symbol
    contract_out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Running triple-exog SV for file {file_path.name} (symbol = {symbol}) ===")

    # 加载数据并做 5 分钟子采样、时间截取
    r, y_star, df, _ = load_single_file(
        file_path,
        contract_code=None,
        start_time='2015-01-01',
        end_time='2025-12-31',
        max_rows=None,
        sample_every=5,
        state_exog_col=None,
        log_exog=True,
    )

    if len(r) <= 100 or len(y_star) <= 100:
        print(f"[{symbol}] Not enough data points (T={len(r)}), skip SV MCMC for this contract.")
        continue

    # 提取 volume/money/avg 三列的对数+标准化外生矩阵
    exog_cols = ['volume', 'money', 'avg']
    exog_mat = prepare_multi_exog(df, exog_cols, log_transform=True, zscore=True, fill_value=0.0)
    if exog_mat is None or exog_mat.shape[0] != len(y_star):
        print(f"[{symbol}] Exogenous matrix missing or misaligned; skip this contract.")
        continue

    # 运行 MCMC（带外生变量）
    mcmc_results = run_mcmc_sv(
        r=r,
        y_star=y_star,
        n_iter=400,
        burn_in=40,
        thin=2,
        rng_seed=2025,
        progress_every=20,
        exog_state=exog_mat,
    )

    extra_info = {
        "file_name": file_path.name,
        "contract_tag": symbol,
        "T": len(r),
        "n_iter": 400,
        "burn_in": 40,
        "thin": 2,
        "sample_every": 5,
        "exog_columns": ",".join(exog_cols),
        "data_dir": str(data_dir),
    }
    save_param_summary(mcmc_results, contract_out_dir, symbol, extra_info=extra_info)

    # 绘制必要图形：潜在波动率、参数后验（含 gamma）、标准化残差、mixture usage
    vol_png = plot_volatility(mcmc_results["h"], df, contract_out_dir, title_suffix=symbol)
    params_png = plot_param_posterior(mcmc_results, contract_out_dir, title_suffix=symbol)
    std_png = plot_standardized_residuals(r, mcmc_results, contract_out_dir, title_suffix=symbol)
    mix_png = None
    if "s" in mcmc_results and len(mcmc_results["s"]) > 0:
        mix_png = plot_mixture_usage(mcmc_results["s"], contract_out_dir, title_suffix=symbol)

    print(
        f"[{symbol}] Figures saved:",
        vol_png,
        params_png,
        std_png,
        mix_png,
    )

    datasets[symbol] = {
        "r": r,
        "y_star": y_star,
        "df": df,
        "exog": exog_mat,
        "mcmc": mcmc_results,
    }

print(f"\nFinished triple-exog SV processing for {len(datasets)} contracts: {list(datasets.keys())}")